# Agentic RAG — Auto Insurance Claim Processing with LangChain

LangChain re-implementation of the smolagents notebook.  
Same tools, same claims workflow, same ChromaDB-backed policy retrieval — but driven by LangChain's `create_tool_calling_agent` / `AgentExecutor`.


## 1 · Install Required Libraries


In [1]:
# Install required packages
# !pip install -q \
#     langchain langchain-openai langchain-community \
#     chromadb PyPDF2 sentence-transformers \
#     pydantic duckduckgo-search

## 2 · Imports & Environment Setup


In [2]:
import csv
import datetime
import json
import os
import re
from typing import List, Optional, Union

import chromadb
import PyPDF2
from pydantic import BaseModel, Field
from sentence_transformers import SentenceTransformer

# LangChain imports
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_community.tools import DuckDuckGoSearchRun

ModuleNotFoundError: No module named 'langchain_core.memory'

In [ ]:
# Load API credentials from config.json
file_name = "../../config.json"
with open(file_name, "r") as file:
    config = json.load(file)
    os.environ["OPENAI_API_KEY"] = config.get("API_KEY")
    os.environ["OPENAI_BASE_URL"] = config.get("OPENAI_API_BASE")

## 3 · Initialize LLM and Embedding Model


In [ ]:
# LLM — GPT-4o-mini via LangChain ChatOpenAI
llm = ChatOpenAI(
    model="gpt-4o-mini",
    openai_api_base=os.environ["OPENAI_BASE_URL"],
    openai_api_key=os.environ["OPENAI_API_KEY"],
)

# Sentence-Transformer embedding model (same as smolagents version)
embedder = SentenceTransformer("all-MiniLM-L6-v2")

## 4 · Pydantic Schemas

`ClaimInfo` · `PolicyQueries` · `PolicyRecommendation` · `ClaimDecision`


In [ ]:
class ClaimInfo(BaseModel):
    """Extracted insurance claim information."""

    claim_number: str
    policy_number: str
    claimant_name: str
    date_of_loss: str
    loss_description: str
    estimated_repair_cost: float
    vehicle_details: Optional[str] = None


class PolicyQueries(BaseModel):
    queries: List[str] = Field(
        default_factory=list,
        description="A list of query strings to retrieve relevant policy sections.",
    )


class PolicyRecommendation(BaseModel):
    """Policy recommendation regarding a given claim."""

    policy_section: str = Field(
        ..., description="The policy section or clause that applies."
    )
    recommendation_summary: str = Field(
        ..., description="A concise summary of coverage determination."
    )
    deductible: Optional[float] = Field(
        None, description="The applicable deductible amount."
    )
    settlement_amount: Optional[float] = Field(
        None, description="Recommended settlement payout."
    )


class ClaimDecision(BaseModel):
    claim_number: str
    covered: bool
    deductible: float
    recommended_payout: float
    notes: Optional[str] = None

## 5 · ChromaDB Setup & Policy Document Loading


In [ ]:
# Initialize ChromaDB in-memory client and collection
chroma_client = chromadb.Client()
collection_name = "auto_insurance_policy"

try:
    collection = chroma_client.get_or_create_collection(name=collection_name)
except Exception as e:
    print(f"Error creating collection: {e}")
    collection = chroma_client.create_collection(name=collection_name)

# Load policy PDF, chunk by double-newline, embed, and store in ChromaDB
policy_file_path = "policy.pdf"

with open(policy_file_path, "rb") as f:
    reader = PyPDF2.PdfReader(f)
    policy_text = ""
    for page in reader.pages:
        policy_text += page.extract_text()

policy_chunks = policy_text.split("\n\n")
chunk_ids = [f"chunk_{i}" for i in range(len(policy_chunks))]
chunk_embeddings = embedder.encode(policy_chunks)

collection.add(documents=policy_chunks, embeddings=chunk_embeddings, ids=chunk_ids)
print(f"Loaded {len(policy_chunks)} policy chunks into ChromaDB.")

## 6 · Define LangChain Tools

Six tools mirror the smolagents originals; LLM calls use `llm.invoke()` instead of `model()`.


In [ ]:
### Tool 1: parse_claim


@tool
def parse_claim(file_path: str) -> str:
    """Parse a claim JSON file and return structured ClaimInfo data as a JSON string.

    Args:
        file_path: Path to the JSON file containing claim data.
    """
    print("[INSIDE TOOL]: parse_claim")
    try:
        with open(file_path, "r") as f:
            data = json.load(f)
        claim_info = ClaimInfo.model_validate(data)
        return claim_info.model_dump_json()
    except Exception as e:
        return f"Error parsing claim: {str(e)}"

In [ ]:
### Tool 2: is_valid_query


@tool
def is_valid_query(query: str) -> str:
    """Check whether the insurance claim is valid against the coverage CSV records.

    Returns a string representation of (bool, reason) so the agent can read the result.

    Args:
        query: The parsed claim data in JSON format (output of parse_claim).
    """
    print("[INSIDE TOOL]: is_valid_query")
    try:
        claim_info = ClaimInfo.model_validate_json(query)

        coverage_data = []
        with open("coverage_data.csv", "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                coverage_data.append(row)
    except Exception as e:
        return f"Error: {str(e)}"

    policy = next(
        (p for p in coverage_data if p["policy_number"] == claim_info.policy_number),
        None,
    )

    if policy is None:
        return str((False, "Policy not found."))

    dues_remaining = policy.get("claim_dues_remaining", "").strip().lower()
    if dues_remaining in ("true", "1", "yes"):
        return str(
            (False, "Due to outstanding payments, the policy is considered invalid.")
        )

    date_loss = datetime.datetime.strptime(str(claim_info.date_of_loss), "%Y-%m-%d")
    date_start = datetime.datetime.strptime(
        str(policy["coverage_start_date"]), "%Y-%m-%d"
    )
    date_end = datetime.datetime.strptime(str(policy["coverage_end_date"]), "%Y-%m-%d")

    if not (date_start <= date_loss <= date_end):
        return str(
            (False, "The date of loss falls outside the policy's coverage period.")
        )

    return str((True, "Valid claim."))

In [ ]:
### Tool 3: generate_policy_queries


@tool
def generate_policy_queries(claim_info_json: str) -> str:
    """Generate 3-5 policy-related search queries based on the claim information.

    Args:
        claim_info_json: JSON string of ClaimInfo data (output of parse_claim).
    """
    print("[INSIDE TOOL]: generate_policy_queries")
    prompt = f"""
    Analyze the following auto insurance claim to identify 3-5 key policy sections to consult:
    - Focus on collision coverage, liability, deductibles, and relevant exclusions or endorsements.
    - Claim Data: {claim_info_json}
    - Return a JSON object with a 'queries' field containing a list of strings,
      e.g., {{"queries": ["query1", "query2", "query3"]}}. Do not include metadata fields.
    """
    try:
        response = llm.invoke([HumanMessage(content=prompt)])
        response_content = response.content
        print(f"[DEBUG]: generate_policy_queries response: {response_content}")

        result = json.loads(response_content)
        if isinstance(result, dict) and "queries" in result:
            queries = result["queries"]
            if isinstance(queries, list) and all(isinstance(q, str) for q in queries):
                return json.dumps(result)
            elif isinstance(queries, list):
                queries = [
                    (q["query"] if isinstance(q, dict) and "query" in q else str(q))
                    for q in queries
                ]
                return json.dumps({"queries": queries})
        return json.dumps({"queries": []})
    except json.JSONDecodeError:
        return f"Error: Invalid JSON response from model"
    except Exception as e:
        return f"Error generating policy queries: {str(e)}"

In [ ]:
### Tool 4: retrieve_policy_text


@tool
def retrieve_policy_text(queries_json: str) -> str:
    """Retrieve relevant policy text chunks from ChromaDB using the provided queries.

    Args:
        queries_json: JSON string with a 'queries' list (output of generate_policy_queries).
    """
    print("[INSIDE TOOL]: retrieve_policy_text")
    try:
        print(f"[DEBUG]: Input queries_json: {queries_json}")
        queries_data = json.loads(queries_json)
        if not isinstance(queries_data, dict) or "queries" not in queries_data:
            return f'Error: Invalid queries_json format, expected {{"queries": [...]}}, got {queries_json}'

        raw_queries = queries_data["queries"]
        if not isinstance(raw_queries, list):
            return f"Error: Queries field must be a list, got {type(raw_queries)}"

        query_strings = []
        for q in raw_queries:
            if isinstance(q, dict) and "query" in q:
                query_strings.append(q["query"])
            elif isinstance(q, str):
                query_strings.append(q)

        validated = PolicyQueries(queries=query_strings)

        policy_texts = []
        for query in validated.queries:
            print(f"[CHROMA SEARCH]: Query: {query}")
            query_embedding = embedder.encode([query])[0]
            results = collection.query(
                query_embeddings=[query_embedding],
                n_results=2,
            )
            policy_texts.extend(results["documents"][0])

        return "\n\n".join(policy_texts)
    except json.JSONDecodeError:
        return "Error: Invalid JSON in queries_json"
    except Exception as e:
        return f"Error retrieving policy text: {str(e)}"

In [ ]:
### Tool 5: generate_recommendation


@tool
def generate_recommendation(claim_info_json: str, policy_text: str) -> str:
    """Generate a structured coverage recommendation using the claim data and retrieved policy text.

    Args:
        claim_info_json: JSON string of ClaimInfo data.
        policy_text: Retrieved policy text from ChromaDB.
    """
    print("[INSIDE TOOL]: generate_recommendation")
    prompt = f"""
    Evaluate the following auto insurance claim against the policy text:
    - Determine if the collision is covered, the deductible, settlement amount, and applicable policy section.
    - Claim Info: {claim_info_json}
    - Policy Text: {policy_text}
    - Return a JSON object matching the following schema:
      {{
        "policy_section": "str",
        "recommendation_summary": "str",
        "deductible": float or null,
        "settlement_amount": float or null
      }}
    - Example:
      {{
        "policy_section": "Exclusions",
        "recommendation_summary": "Claim denied due to business use exclusion",
        "deductible": null,
        "settlement_amount": 0.0
      }}
    - Do not use fields like 'recommendation', 'coverage_evaluation', or 'reason'.
    """
    try:
        response = llm.invoke([HumanMessage(content=prompt)])
        response_content = response.content
        print(f"[DEBUG]: generate_recommendation response: {response_content}")

        result = json.loads(response_content)
        PolicyRecommendation.model_validate(result)  # validate schema
        return response_content
    except json.JSONDecodeError:
        return f"Error: Invalid JSON response from model"
    except Exception as e:
        return f"Error generating recommendation: {str(e)}"

In [ ]:
### Tool 6: finalize_decision


@tool
def finalize_decision(claim_info_json: str, recommendation_json: str) -> str:
    """Produce a final ClaimDecision JSON from the claim info and policy recommendation.

    Args:
        claim_info_json: JSON string of ClaimInfo data.
        recommendation_json: JSON string of PolicyRecommendation data.
    """
    print("[INSIDE TOOL]: finalize_decision")
    print(f"[DEBUG]: recommendation_json: {recommendation_json}")
    try:
        claim_info = ClaimInfo.model_validate_json(claim_info_json)
        rec_data = json.loads(recommendation_json)
        if not isinstance(rec_data, dict):
            return f"Error: recommendation_json must be a JSON object, got {type(rec_data)}"

        # Normalise unexpected keys to the PolicyRecommendation schema
        if "policy_section" not in rec_data:
            recommendation_text = rec_data.get(
                "recommendation",
                rec_data.get("reason", rec_data.get("coverage_evaluation", "")),
            )
            policy_match = re.search(
                r"(Part [A-D]|\bExclusions\b|\bCollision Coverage\b)",
                recommendation_text,
                re.IGNORECASE,
            )
            rec_data["policy_section"] = (
                policy_match.group(0) if policy_match else "Unknown Section"
            )
        if "recommendation_summary" not in rec_data:
            rec_data["recommendation_summary"] = rec_data.get(
                "recommendation",
                rec_data.get(
                    "reason", rec_data.get("coverage_evaluation", "No summary provided")
                ),
            )
        rec_data.setdefault("deductible", None)
        rec_data.setdefault("settlement_amount", 0.0)

        rec = PolicyRecommendation.model_validate(rec_data)
        covered = "covered" in rec.recommendation_summary.lower() or (
            rec.settlement_amount is not None and rec.settlement_amount > 0
        )
        deductible = rec.deductible if rec.deductible is not None else 0.0
        recommended_payout = rec.settlement_amount if rec.settlement_amount else 0.0

        decision = ClaimDecision(
            claim_number=claim_info.claim_number,
            covered=covered,
            deductible=deductible,
            recommended_payout=recommended_payout,
            notes=rec.recommendation_summary,
        )
        return decision.model_dump_json()
    except Exception as e:
        return f"Error finalizing decision: {str(e)}"

## 7 · Build the Agentic RAG Pipeline with LangChain

`create_tool_calling_agent` + `AgentExecutor` replace Smolagents' `ToolCallingAgent`.  
`DuckDuckGoSearchRun` replaces `WebSearchTool` for repair-cost estimation.


In [ ]:
system_prompt = """
You are an expert insurance claim-processing agent specialising in auto insurance.
Follow this MANDATORY multi-step workflow in order:

1. Parse the claim file → call parse_claim with the file path.
2. Validate the claim → call is_valid_query with the JSON from step 1.
   - If the result contains False, STOP and report an invalid claim immediately.
3. Generate policy queries → call generate_policy_queries with the claim JSON.
4. Retrieve policy text → call retrieve_policy_text with the queries JSON.
5. Estimate repair cost → call duckduckgo_search to look up typical market repair prices for the
   described damage. Compare with the claimed amount. If the amount is unrealistic / inflated,
   report an invalid (inflated) claim and stop.
6. Generate recommendation → call generate_recommendation with claim JSON + policy text.
7. Finalise decision → call finalize_decision with claim JSON + recommendation JSON.
8. Output the final decision in this exact format:

Claim Processing Summary:
- Claim Number: <claim_number>
- Coverage Status: Valid  OR  Invalid
- Deductible: $<amount>
- Recommended Payout: $<amount>
- Notes: <notes>
"""

# Prompt template for the tool-calling agent
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

# Tool list (DuckDuckGoSearchRun replaces smolagents WebSearchTool)
web_search = DuckDuckGoSearchRun()

tools = [
    parse_claim,
    is_valid_query,
    web_search,
    generate_policy_queries,
    retrieve_policy_text,
    generate_recommendation,
    finalize_decision,
]

# Build agent and executor
agent = create_tool_calling_agent(llm, tools, prompt)
claim_processing_agent = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=20,
)

print("Agent ready.")

## 8 · Run Queries Against the Agentic RAG System

### Run 1 · Policy not found → claim should be rejected

`PL-1` is not in `coverage_data.csv`, so the agent should stop at the validation step and report an invalid claim.


In [ ]:
# Run 1 — invalid policy number
ema_claim_data = {
    "claim_number": "CLAIM-00001",
    "policy_number": "PL-1",  # not in coverage_data.csv
    "claimant_name": "Ema Johnson",
    "date_of_loss": "2022-01-21",
    "loss_description": "Rear-ended by another driver at a red light, resulting in bumper damage and mild whiplash.",
    "estimated_repair_cost": 150.0,
    "vehicle_details": "2022 Honda City",
}

with open("ema.json", "w") as f:
    json.dump(ema_claim_data, f)

result1 = claim_processing_agent.invoke(
    {"input": "Process the insurance claim from file: ema.json"}
)
print("\n=== Run 1 Result ===")
print(result1["output"])

### Run 2 · Inflated repair cost → claim should be rejected

A bumper replacement on a Honda City should not cost $5 000; the agent should detect the mismatch via a web search and reject the claim.


In [ ]:
# Run 2 — inflated estimated repair cost
ema_claim_data = {
    "claim_number": "CLAIM-00001",
    "policy_number": "PN-1",
    "claimant_name": "Ema Johnson",
    "date_of_loss": "2022-01-21",
    "loss_description": "Rear-ended by another driver at a red light, resulting in bumper damage and mild whiplash.",
    "estimated_repair_cost": 5000.0,  # unrealistically high for a bumper
    "vehicle_details": "2022 Honda City",
}

with open("ema2.json", "w") as f:
    json.dump(ema_claim_data, f)

result2 = claim_processing_agent.invoke(
    {"input": "Process the insurance claim from file: ema2.json"}
)
print("\n=== Run 2 Result ===")
print(result2["output"])

### Run 3 · Full happy-path flow → claim should be approved

Valid policy, reasonable cost, date within coverage window — the full pipeline should run and return an approval decision.


In [ ]:
# Run 3 — valid claim, realistic cost, full pipeline
ema_claim_data = {
    "claim_number": "CLAIM-00001",
    "policy_number": "PN-1",
    "claimant_name": "Ema Johnson",
    "date_of_loss": "2022-01-21",
    "loss_description": "The car was rear ended by a truck when parked at the office. The rear bumper was destroyed in the process.",
    "estimated_repair_cost": 550.0,
    "vehicle_details": "2022 Honda City",
}

with open("ema3.json", "w") as f:
    json.dump(ema_claim_data, f)

result3 = claim_processing_agent.invoke(
    {"input": "Process the insurance claim from file: ema3.json"}
)
print("\n=== Run 3 Result ===")
print(result3["output"])